# Distribución de almacén y preparación de pedidos

## Tipo de problema

Asignar ubicaciones a las referencias de producto (SKUs) para minimizar la distancia recorrida por los operarios o robots durante la preparación de pedidos (picking).

## Formas de resolverlo

Los nombres de la primera columna coinciden con la [tabla del README](../README.md#formas-de-resolverlo).

| Enfoque (README) | Aplicación en este problema | Uso en este ejercicio |
|---|---|---|
| Programación lineal | Formular asignación lineal si el coste solo depende de cada producto y su ubicación; también admite algoritmos específicos como el húngaro. | Alternativa |
| No lineal o cuadrática | Formular asignación cuadrática (QAP) cuando importan las parejas de productos. | Alternativa |
| Heurísticas y metaheurísticas | Mejorar ubicaciones por intercambios, tabú o recocido simulado. | Alternativa |
| Enumeración | Resolver exactamente el QAP pequeño comparando las 24 asignaciones. | **Principal** |

**Enfoque utilizado:** Enumeración.

**Técnica concreta:** Evaluar las 24 asignaciones del modelo cuadrático de ubicación, contando pedidos individuales y de parejas.

## Ejercicio propuesto

Asignar cuatro productos A-D a cuatro posiciones del almacén.  
Depósito (0,0); posiciones (1,0),(2,0),(1,2),(3,2), una referencia por posición.  
Pedidos de una sola referencia: A=8, B=6, C=3, D=2. Pedidos de dos referencias:  
AB=4, AC=1, AD=0, BC=2, BD=1, CD=3. Todos son conteos de un día.  
Cada pedido sale del depósito y regresa; distancia Manhattan sin obstáculos.  
Una pareja i,j recorre depósito-i-j-depósito (ambos órdenes cuestan igual).  
Objetivo: mínima distancia diaria. Método exacto: evaluar las 4! asignaciones.  
El término entre parejas hace que sea asignación cuadrática, no solo ordenar  
productos por frecuencia. No se modelan congestión ni pedidos de más de dos SKU.

## Método elegido para este ejercicio

**Enfoque utilizado:** Enumeración.

Se comparan las 24 asignaciones posibles. Se suma el recorrido de pedidos individuales y de parejas. Los pedidos de dos referencias introducen costes entre posiciones: es asignación cuadrática. Se usa distancia Manhattan sin obstáculos.

## Implementación

Ejecuta las celdas de código en orden. Solo se utiliza la biblioteca estándar de Python.

In [1]:
from itertools import permutations


def resolver():
    posiciones = ((1, 0), (2, 0), (1, 2), (3, 2))
    individuales = (8, 6, 3, 2)
    parejas = {(0, 1): 4, (0, 2): 1, (0, 3): 0,
               (1, 2): 2, (1, 3): 1, (2, 3): 3}

    def distancia(a, b):
        return sum(abs(x - y) for x, y in zip(a, b))

    def evaluar(asignacion):
        puntos = [posiciones[k] for k in asignacion]
        al_deposito = [distancia((0, 0), p) for p in puntos]
        total = sum(2 * n * d for n, d in zip(individuales, al_deposito))
        total += sum(n * (al_deposito[i] + distancia(puntos[i], puntos[j]) + al_deposito[j])
                     for (i, j), n in parejas.items())
        return total

    mejor = min(permutations(range(4)), key=evaluar)
    assert len(set(mejor)) == 4
    return evaluar(mejor), dict(zip('ABCD', (posiciones[k] for k in mejor)))

## Resolución y resultados

In [2]:
total, ubicaciones = resolver()
for producto, posicion in ubicaciones.items():
    print(f'Producto {producto}: posición {posicion}')
print(f'Distancia diaria mínima: {total} unidades de distancia')

Producto A: posición (1, 0)
Producto B: posición (2, 0)
Producto C: posición (1, 2)
Producto D: posición (3, 2)
Distancia diaria mínima: 156 unidades de distancia


## Interpretación

Ubicar **A en (1,0), B en (2,0), C en (1,2) y D en (3,2)**. La distancia diaria mínima es **156 unidades de distancia**. La solución tiene en cuenta tanto frecuencia individual como referencias pedidas juntas.

**Alcance:** los datos son sintéticos. Las garantías y supuestos se refieren a esta instancia; al aumentar su tamaño, la enumeración puede resultar costosa.